# Web Scraping — Okezone

Scraper artikel berita **keluhan / masalah publik** dari **Okezone**.

Pipeline notebook ini (struktur seragam dengan 7 scraper lainnya):

1. Import library.
2. **Konfigurasi** — BASE_URL, blok keyword, cap per keyword/sumber, rentang tanggal, `TEST_MODE`, `SELECTORS`.
3. **HTTP helper** — session, retry + exponential backoff, jitter delay.
4. **Cek `robots.txt`**.
5. **Kumpulkan link** per keyword (round-robin, pagination).
6. **Parse artikel** — JSON-LD / meta tag diutamakan, CSS selector sebagai fallback.
7. **Pipeline utama** — dedup, filter tanggal & panjang, checkpoint, log error.
8. **Kuota netral** — KEYWORDS_NEUTRAL untuk kelas 'Bukan Pengaduan'.
9. **Bersihkan & simpan** ke `../data/raw/okezone_raw.csv` (utf-8-sig).
10. **Ringkasan + cek kualitas**.

> **Cara pakai:** set `TEST_MODE = True` dulu (2 keyword, 1–2 halaman, maks 10 artikel).
> Kalau CSV terbentuk dengan kolom benar & konten tidak kosong, set `TEST_MODE = False`,
> lalu **Restart & Run All**.

> **Catatan situs:** Okezone MENGIZINKAN pencarian: `search.okezone.com/search?q=<kw>&page=<n>` (robots.txt tidak melarang /search). Pagination lewat param `page` (diverifikasi: page=2 beda dari page=1). Hasil 10 link/halaman di container `.list-news`. URL artikel memuat tanggal (`/read/YYYY/MM/DD/...`) sehingga filter tanggal 5 tahun bekerja; untuk menjangkau artikel lama, pagination harus cukup dalam.

> ⚠️ Selector di `SELECTORS` bertanda **PERLU DIVERIFIKASI** — belum diuji terhadap
> situs live. Verifikasi lewat Inspect Element saat running (lihat sel penutup).

## 1. Import Library

In [1]:
# Library yang dipakai (pasang sendiri di environment Anda bila belum ada):
#   requests, beautifulsoup4, lxml, pandas, tqdm
# Sisanya modul standar Python.
import os
import re
import csv
import time
import json
import random
import logging
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse
from urllib import robotparser

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("scraper_okezone")

c:\Users\62895\anaconda3\envs\text_mining\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Konfigurasi

Semua parameter yang bisa diubah ada di sel ini: identitas situs, blok keyword
(seragam di semua notebook), cap jumlah, rentang tanggal, dan `SELECTORS`.

In [2]:
SOURCE_NAME = 'okezone'
BASE_URL = 'https://news.okezone.com'
SEARCH_URL = 'https://search.okezone.com/search'  # [CEK SAAT RUNNING] endpoint pencarian
# Template parameter pencarian: {keyword} & {page} disubstitusi saat request.
SEARCH_PARAMS = {
    'q': '{keyword}',
    'page': '{page}',
}  # [CEK SAAT RUNNING]
PAGE_PARAM = 'page'
ARTICLE_URL_RE = re.compile(r'https?://[a-z]+\.okezone\.com/read/\d{4}/\d{2}/\d{2}/\d+/\d+/.+')  # [CEK SAAT RUNNING] pola URL artikel
IS_DYNAMIC = False      # konten dirender JS? (Kumparan=True)
MULTIPAGE_ARTICLE = False  # artikel berhalaman?
PAGE_ALL_PARAM = ""
USE_INDEX = False  # True: pakai halaman indeks/kanal (bukan /search)
INDEX_PAGES = []
INDEX_PAGE_PARAM = 'page'  # nama query-param halaman pada index ("" = tanpa pagination)
USE_DATE_INDEX = False  # True: telusuri arsip per TANGGAL mundur sampai DATE_FROM
DATE_INDEX_URL_FMT = ''  # pola URL indeks tanggal, mis. ".../indeks/{yyyy}/{mm}/{dd}"

# ======================================================================
# DAFTAR KEYWORD (seragam di semua scraper) -- WAJIB
# ======================================================================
KEYWORDS_BY_CATEGORY = {
    "Sampah": ["sampah menumpuk", "tumpukan sampah", "sampah berserakan", "tempat pembuangan sampah liar", "sampah tidak diangkut", "bau sampah warga", "buang sampah sembarangan", "TPS penuh"],
    "Jalan": ["jalan rusak", "jalan berlubang", "jalan amblas", "perbaikan jalan warga", "jalan bergelombang", "aspal rusak", "jembatan rusak", "trotoar rusak"],
    "Saluran Air/Drainase": ["drainase tersumbat", "saluran air mampet", "got tersumbat", "selokan meluap", "gorong-gorong tersumbat", "air tergenang drainase buruk", "normalisasi saluran air"],
    "Fasos/Fasum": ["fasilitas umum rusak", "fasum rusak", "taman rusak", "halte rusak", "jembatan penyeberangan rusak", "toilet umum rusak", "fasilitas sosial terbengkalai", "pagar taman rusak"],
    "Ketertiban Umum": ["pkl liar", "parkir liar", "pedagang kaki lima trotoar", "bangunan liar", "juru parkir liar", "pungli", "premanisme warga resah", "kebisingan warga mengeluh", "balap liar warga resah"],
    "Lampu/PJU": ["lampu jalan mati", "penerangan jalan umum mati", "PJU mati", "jalan gelap warga", "lampu jalan rusak", "lampu lalu lintas mati"],
    "Banjir": ["banjir", "banjir rendam permukiman", "banjir rob", "genangan air", "warga terdampak banjir", "banjir kiriman", "luapan sungai"],
    "Transportasi/Lalu Lintas": ["macet", "kemacetan parah warga", "angkutan umum buruk", "kecelakaan jalan rusak", "lampu merah rusak macet", "transportasi umum keluhan", "parkir sembarangan macet", "ojek online keluhan"],
}

KEYWORDS_GENERAL_COMPLAINT = ["warga mengeluh", "warga keluhkan", "warga resah", "aduan warga", "pengaduan masyarakat", "keluhan warga", "warga protes", "warga minta pemerintah", "viral keluhan warga", "laporan warga", "warga lapor", "belum ditangani pemerintah"]

# Untuk kelas "Bukan Pengaduan": kata kunci netral / bukan keluhan
KEYWORDS_NEUTRAL = ["politik", "olahraga", "ekonomi bisnis", "hiburan selebriti", "teknologi", "kesehatan gaya hidup", "pendidikan prestasi", "wisata kuliner", "otomotif", "internasional"]

# Daftar keyword keluhan (urut, dedup) -> dipakai round-robin di pipeline.
KEYWORDS = list(dict.fromkeys(
    [kw for kws in KEYWORDS_BY_CATEGORY.values() for kw in kws] + KEYWORDS_GENERAL_COMPLAINT
))

# Peta keyword -> kategori asal (petunjuk awal pelabelan, BUKAN label final).
KEYWORD_TO_CATEGORY = {kw: cat for cat, kws in KEYWORDS_BY_CATEGORY.items() for kw in kws}
for kw in KEYWORDS_GENERAL_COMPLAINT:
    KEYWORD_TO_CATEGORY.setdefault(kw, "General")
for kw in KEYWORDS_NEUTRAL:
    KEYWORD_TO_CATEGORY[kw] = "Neutral"

# Istilah INTI per kategori untuk INDEX MODE: artikel dianggap relevan bila judul/URL
# memuat salah satu istilah ini. Lebih longgar dari frasa penuh (mis. judul 'Sampah
# Menggunung di Pasar' tetap tertangkap kategori Sampah) tapi tetap on-topik.
CATEGORY_CORE_TERMS = {
    "Sampah": ["sampah", "tps", "tempat pembuangan"],
    "Jalan": ["jalan rusak", "jalan berlubang", "jalan amblas", "aspal rusak", "jembatan rusak", "trotoar rusak", "jalan bergelombang"],
    "Saluran Air/Drainase": ["drainase", "gorong-gorong", "selokan", "saluran air", "got mampet", "got tersumbat"],
    "Fasos/Fasum": ["fasilitas umum", "fasum", "halte rusak", "taman rusak", "toilet umum", "jpo rusak"],
    "Ketertiban Umum": ["pkl liar", "parkir liar", "bangunan liar", "pungli", "premanisme", "balap liar", "juru parkir"],
    "Lampu/PJU": ["lampu jalan", "penerangan jalan", "pju", "lampu lalu lintas", "jalan gelap"],
    "Banjir": ["banjir", "rob", "genangan", "luapan sungai", "kebanjiran"],
    "Transportasi/Lalu Lintas": ["macet", "kemacetan", "angkutan umum", "lalu lintas", "transjakarta", "krl", "transportasi umum"],
}
# Istilah inti untuk keluhan umum (General).
GENERAL_CORE_TERMS = ["warga mengeluh", "warga keluhkan", "warga resah", "aduan warga", "pengaduan", "keluhan warga", "warga protes", "warga lapor", "dikeluhkan warga", "keluhkan"]

# ======================================================================
# Parameter pengambilan (poin A spesifikasi)
# ======================================================================
TEST_MODE = False             # True: uji kecil (maks 10); False: run penuh (target 5000)

# --- Target volume (diperbesar untuk target TOTAL >= 5000 artikel lintas 8 situs) ---
# 8 situs x ~750 = 6000 kandidat -> setelah dedup/filter diharapkan >= 5000.
# Tempo sering memblokir request (403); bila gagal, situs lain menutupi kuotanya.
MAX_ARTICLES_PER_KEYWORD = 120 # cap artikel per keyword (search mode)
MAX_ARTICLES_PER_SOURCE = 750  # cap total artikel per situs
NEUTRAL_RATIO = 0.12           # ~12% target diisi artikel netral
MAX_PAGES = 40                 # batas halaman pencarian per keyword / kanal (run penuh)

# Rentang tanggal: 5 TAHUN terakhir (60 bulan) agar volume besar untuk target 5000.
# Ubah pengali bulan di bawah bila ingin rentang lain (mis. 12 = 1 tahun).
DATE_TO = datetime.now(timezone.utc)
DATE_FROM = DATE_TO - timedelta(days=365 * 5)  # 5 tahun terakhir

MIN_CONTENT_CHARS = 200       # buang artikel yang isinya < 200 karakter

# --- Etika & robustness ---
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
)
DELAY_MIN, DELAY_MAX = 1.5, 3.5  # jitter antar request (detik)
TIMEOUT = 20
MAX_RETRIES = 3
BACKOFF_FACTOR = 2.0

# --- Output (semua path RELATIF ke folder notebook) ---
RAW_DIR = Path("..") / "data" / "raw"
LOG_DIR = RAW_DIR / "logs"
OUTPUT_PATH = RAW_DIR / f"{SOURCE_NAME}_raw.csv"
CHECKPOINT_PATH = RAW_DIR / f"{SOURCE_NAME}_checkpoint.csv"
ERROR_LOG_PATH = LOG_DIR / f"{SOURCE_NAME}_errors.csv"
CHECKPOINT_EVERY = 10
RAW_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# --- SELECTORS (satu tempat; PERLU DIVERIFIKASI terhadap situs live) ---
SELECTORS = {
    'search_link': [
        'div.list-news a.box-img[href]',
        'div.box-text a.title[href]',
        'div.list-news a[href]',
        'a.title[href]',
    ],
    'title': [
        'h1.title',
        "h1[class*='title']",
        'h1',
    ],
    'date': [
        'time[datetime]',
        '.fast-reading-time',
        '.date',
        "div[class*='date']",
    ],
    'author': [
        '.namerelates',
        '.reporter',
        '.author',
        "div[class*='author']",
    ],
    'section': [
        "meta[property='article:section']",
    ],
    'body': [
        'div.c-detail.read',
        'div.c-detail',
        '.detail-content',
        'section.article .c-detail',
    ],
    'junk': [
        'script',
        'style',
        '.baca',
        'figure',
        '.ads',
        '.baca-juga',
        '.tag',
        '.related',
        '.paralaks-ads',
        '.adsbygoogle',
    ],
}

# --- Resolusi mode aktif ---
if TEST_MODE:
    # SEARCH MODE: cukup 2 keyword. INDEX MODE: cocokkan SEMUA keyword ke halaman kanal
    # (agar smoke test tetap menemukan artikel), tapi tetap batasi 10 artikel.
    ACTIVE_KEYWORDS = KEYWORDS if (USE_INDEX or USE_DATE_INDEX) else KEYWORDS[:2]
    ACTIVE_MAX_PAGES = 2
    ACTIVE_TARGET = 10                  # maks 10 artikel total
    ACTIVE_PER_KEYWORD = 10
    ACTIVE_NEUTRAL_QUOTA = 0
    DATE_INDEX_MAX_DAYS = 7             # date-index: cek 7 hari terakhir saja saat uji
else:
    ACTIVE_KEYWORDS = KEYWORDS
    ACTIVE_MAX_PAGES = MAX_PAGES
    ACTIVE_TARGET = MAX_ARTICLES_PER_SOURCE
    ACTIVE_PER_KEYWORD = MAX_ARTICLES_PER_KEYWORD
    ACTIVE_NEUTRAL_QUOTA = int(MAX_ARTICLES_PER_SOURCE * NEUTRAL_RATIO)
    DATE_INDEX_MAX_DAYS = 365 * 5      # date-index: telusuri sampai 5 tahun ke belakang

OUTPUT_COLUMNS = [
    "source", "url", "title", "published_date", "category_section",
    "author", "content", "scraped_at", "keyword_used", "keyword_category",
]

print(f"Mode               : {'TEST' if TEST_MODE else 'FULL'}")
print(f"Keyword aktif      : {len(ACTIVE_KEYWORDS)}")
print(f"Target artikel     : {ACTIVE_TARGET} (netral ~{ACTIVE_NEUTRAL_QUOTA})")
print(f"Per keyword maks   : {ACTIVE_PER_KEYWORD}")
print(f"Rentang tanggal    : {DATE_FROM.date()} .. {DATE_TO.date()}")
print(f"Output             : {OUTPUT_PATH}")

Mode               : FULL
Keyword aktif      : 73
Target artikel     : 750 (netral ~90)
Per keyword maks   : 120
Rentang tanggal    : 2021-10-21 .. 2026-10-20
Output             : ..\data\raw\okezone_raw.csv


## 3. HTTP Helper (session + retry + backoff + jitter)

In [3]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": USER_AGENT,
    "Accept-Language": "id-ID,id;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
})

def polite_sleep():
    """Jeda acak (jitter) antar request agar tidak membebani server."""
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def fetch(url, params=None):
    """GET dengan retry + exponential backoff. Tangani 403/429/5xx. Return Response|None."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = SESSION.get(url, params=params, timeout=TIMEOUT)
        except requests.RequestException as exc:
            wait = BACKOFF_FACTOR ** attempt
            logger.warning("Req error %s (attempt %d/%d). Backoff %.1fs", exc, attempt, MAX_RETRIES, wait)
            time.sleep(wait)
            continue
        if resp.status_code == 200:
            return resp
        if resp.status_code == 403:
            logger.error("HTTP 403 Forbidden di %s -- situs menolak request. Skip.", url)
            return None
        if resp.status_code in (429, 500, 502, 503, 504):
            wait = BACKOFF_FACTOR ** attempt
            logger.warning("HTTP %s di %s (attempt %d/%d). Backoff %.1fs", resp.status_code, url, attempt, MAX_RETRIES, wait)
            time.sleep(wait)
            continue
        logger.warning("HTTP %s di %s. Skip.", resp.status_code, url)
        return None
    logger.error("Gagal setelah %d percobaan: %s", MAX_RETRIES, url)
    return None

def select_first(soup, selector_list, attr=None):
    """Coba tiap selector di list; kembalikan teks (atau atribut) dari yang pertama cocok."""
    for sel in selector_list:
        el = soup.select_one(sel)
        if el is not None:
            if attr:
                val = el.get(attr, "")
                if val:
                    return val
            else:
                txt = el.get_text(' ', strip=True)
                if txt:
                    return txt
    return ""

## 4. Cek `robots.txt`

Baca `robots.txt` dengan `urllib.robotparser`, tampilkan apakah path pencarian &
artikel diizinkan, dan hormati hasilnya.

In [4]:
def check_robots(base_url, user_agent, test_paths):
    """Cek izin akses path. Return (bool_semua_boleh, dict_hasil)."""
    rp = robotparser.RobotFileParser()
    robots_url = urljoin(base_url, "/robots.txt")
    try:
        rp.set_url(robots_url)
        rp.read()
    except Exception as exc:
        logger.warning("Gagal baca robots.txt (%s): %s. Lanjut hati-hati.", robots_url, exc)
        return True, {}
    results = {p: rp.can_fetch(user_agent, urljoin(base_url, p)) for p in test_paths}
    for path, allowed in results.items():
        logger.info("robots.txt %-30s -> %s", path, "ALLOW" if allowed else "DISALLOW")
    return all(results.values()), results

search_path = urlparse(SEARCH_URL).path or '/' if SEARCH_URL else '/'
if USE_DATE_INDEX:
    # Mode date-index: cek root + path indeks tanggal contoh.
    _ex = DATE_INDEX_URL_FMT.format(yyyy='2024', mm='01', dd='01') if DATE_INDEX_URL_FMT else BASE_URL
    check_paths = [urlparse(_ex).path or '/', '/']
elif USE_INDEX:
    # Mode index: cek path kanal yang benar-benar dipakai.
    check_paths = [urlparse(p).path or '/' for p in INDEX_PAGES] + ['/']
else:
    check_paths = [search_path, '/']
robots_ok, robots_detail = check_robots(BASE_URL, USER_AGENT, check_paths)
if not robots_ok:
    logger.warning("Sebagian path dilarang robots.txt untuk %s. Tinjau sebelum run penuh.", SOURCE_NAME)
if USE_DATE_INDEX:
    logger.info("USE_DATE_INDEX=True: notebook ini menelusuri arsip per tanggal (/indeks/YYYY/MM/DD), bukan /search.")
elif USE_INDEX:
    logger.info("USE_INDEX=True: notebook ini memakai halaman kanal, BUKAN /search (dihormati sesuai robots.txt).")

2026-10-20 23:54:03,129 [INFO] robots.txt /search                        -> ALLOW
2026-10-20 23:54:03,131 [INFO] robots.txt /                              -> ALLOW


## 5. Kumpulkan Link Artikel

Dua mode (dipilih otomatis lewat `USE_INDEX` di sel Konfigurasi):

- **SEARCH MODE** (`USE_INDEX=False`): iterasi keyword **round-robin** di endpoint
  `/search`. Dipakai untuk situs yang mengizinkan pencarian (detikcom, kompas, tempo, merdeka).
- **INDEX MODE** (`USE_INDEX=True`): untuk situs yang **men-DISALLOW `/search`** di
  `robots.txt` (cnnindonesia, kumparan, liputan6, tribunnews). Mengambil link dari
  halaman kanal `INDEX_PAGES` yang diizinkan, lalu **memfilter** artikel yang judul/URL-nya
  cocok dengan keyword (`match_keywords`). Ini menghormati robots.txt tanpa mengarang jalan pintas.

> `# [CEK SAAT RUNNING]`: `SELECTORS['search_link']`, `INDEX_PAGES`, & parameter pagination paling mungkin berubah.

In [5]:
def build_search_params(keyword, page):
    """Substitusi template SEARCH_PARAMS untuk satu keyword & halaman."""
    params = {}
    for k, v in SEARCH_PARAMS.items():
        if v == '{keyword}':
            params[k] = keyword
        elif v == '{page}':
            params[k] = page
        else:
            params[k] = v
    return params

def extract_links_from_search(html):
    """Ambil set URL artikel dari satu halaman hasil pencarian."""
    soup = BeautifulSoup(html, "lxml")
    links = set()
    for sel in SELECTORS['search_link']:  # [CEK SAAT RUNNING]
        for a in soup.select(sel):
            href = a.get("href", "")
            if not href:
                continue
            href = urljoin(BASE_URL, href.split('#')[0])
            if ARTICLE_URL_RE.match(href):
                links.add(href.split('?')[0])
    # Fallback: semua <a> yang cocok pola URL artikel.
    if not links:
        for a in soup.find_all("a", href=True):
            href = urljoin(BASE_URL, a['href'].split('#')[0])
            if ARTICLE_URL_RE.match(href):
                links.add(href.split('?')[0])
    return links

import unicodedata

def _norm(text):
    """Lowercase + buang aksen untuk pencocokan keyword yang longgar."""
    text = unicodedata.normalize('NFKD', text or '')
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text.lower()

def _word_in(word, text):
    """True bila `word` muncul sebagai KATA utuh di text (bukan substring).
    Mencegah false match seperti rob di dalam problem atau pju di tengah kata."""
    if len(word) <= 2:
        return False  # token terlalu pendek -> abaikan
    return re.search(r'(?<![a-z])' + re.escape(word) + r'(?![a-z])', text) is not None

def _term_matches(term, text):
    """Semua kata dalam `term` harus muncul sebagai kata utuh."""
    words = _norm(term).split()
    return bool(words) and all(_word_in(w, text) for w in words)

def match_keywords(text):
    """INDEX MODE: cocokkan artikel ke kategori via istilah INTI dengan pencocokan
    KATA UTUH (word-boundary), bukan substring. Return list of (keyword_label, category)."""
    t = _norm(text)
    out = []
    seen = set()
    # Istilah netral (untuk kuota 'Bukan Pengaduan').
    if CURRENT_IS_NEUTRAL:
        for kw in KEYWORDS_NEUTRAL:
            if _term_matches(kw, t) and kw not in seen:
                seen.add(kw); out.append((kw, 'Neutral'))
        return out
    # Istilah inti per kategori keluhan.
    for cat, terms in CATEGORY_CORE_TERMS.items():
        for term in terms:
            if _term_matches(term, t):
                if term not in seen:
                    seen.add(term); out.append((term, cat))
                break  # satu match per kategori cukup
    # Istilah keluhan umum.
    for term in GENERAL_CORE_TERMS:
        if _term_matches(term, t) and term not in seen:
            seen.add(term); out.append((term, 'General'))
    return out

def collect_via_search(keywords, max_pages, per_keyword_cap, overall_cap):
    """Round-robin antar keyword via endpoint /search. Return dict {url: {keywords,category}}."""
    collected = {}
    per_kw_count = {kw: 0 for kw in keywords}
    page_cursor = {kw: 1 for kw in keywords}
    exhausted = set()
    while len(collected) < overall_cap * 2 and len(exhausted) < len(keywords):
        for kw in keywords:
            if kw in exhausted or per_kw_count[kw] >= per_keyword_cap:
                exhausted.add(kw)
                continue
            page = page_cursor[kw]
            if page > max_pages:
                exhausted.add(kw)
                continue
            resp = fetch(SEARCH_URL, params=build_search_params(kw, page))
            polite_sleep()
            page_cursor[kw] += 1
            if resp is None:
                exhausted.add(kw)
                continue
            page_links = extract_links_from_search(resp.text)
            if not page_links:
                exhausted.add(kw)
                continue
            added = 0
            for url in page_links:
                if per_kw_count[kw] >= per_keyword_cap:
                    break
                entry = collected.setdefault(url, {'keywords': set(), 'category': KEYWORD_TO_CATEGORY.get(kw, 'General')})
                if kw not in entry['keywords']:
                    entry['keywords'].add(kw)
                    per_kw_count[kw] += 1
                    added += 1
            logger.info("[%s] page %d: +%d (total unik %d)", kw, page, added, len(collected))
            if len(collected) >= overall_cap * 2:
                break
    return collected

def extract_links_with_titles(html):
    """INDEX MODE: ambil (url, anchor_text) dari halaman kanal."""
    soup = BeautifulSoup(html, 'lxml')
    out = {}
    for a in soup.find_all('a', href=True):
        href = urljoin(BASE_URL, a['href'].split('#')[0])
        if ARTICLE_URL_RE.match(href):
            clean = href.split('?')[0]
            out.setdefault(clean, a.get_text(' ', strip=True))
    return out

def collect_via_index(keywords, max_pages, per_keyword_cap, overall_cap):
    """INDEX MODE: ambil dari INDEX_PAGES (kanal). Link yang judulnya cocok keyword
    langsung di-tag kategori. Link lain disimpan sebagai '__PENDING__' (sampai batas
    anggaran) agar RELEVANSI ditentukan dari ISI artikel saat parse_article -- ini
    menaikkan recall karena banyak artikel keluhan berjudul tak mengandung kata inti."""
    collected = {}
    per_cat_count = {}
    pending_budget = overall_cap * 3  # anggaran link '__PENDING__' untuk dicek via konten
    for base_page in INDEX_PAGES:
        for page in range(1, max_pages + 1):
            if INDEX_PAGE_PARAM:
                resp = fetch(base_page, params={INDEX_PAGE_PARAM: page})
            else:
                resp = fetch(base_page) if page == 1 else None
            polite_sleep()
            if resp is None:
                break
            found = extract_links_with_titles(resp.text)
            if not found:
                break
            added = 0
            for url, anchor in found.items():
                if url in collected:
                    continue
                hits = match_keywords(f'{anchor} {url}')  # list of (term, category)
                if hits:
                    term, category = hits[0]
                    if per_cat_count.get(category, 0) >= per_keyword_cap:
                        continue
                    collected[url] = {'keywords': set(t for t, _ in hits), 'category': category}
                    per_cat_count[category] = per_cat_count.get(category, 0) + 1
                    added += 1
                elif pending_budget > 0:
                    # Simpan untuk dicek dari isi artikel nanti (kategori ditentukan di parse_article).
                    collected[url] = {'keywords': set(), 'category': '__PENDING__'}
                    pending_budget -= 1
                    added += 1
            logger.info("[index %s] page %d: +%d (total %d, pending_budget %d)", base_page.rsplit("/", 1)[-1], page, added, len(collected), pending_budget)
            if len(collected) >= overall_cap * 4:
                return collected
    return collected

def collect_via_date_index(keywords, max_pages, per_keyword_cap, overall_cap):
    """DATE-INDEX MODE: telusuri arsip per TANGGAL (DATE_INDEX_URL_FMT) dari DATE_TO
    mundur hari demi hari sampai DATE_FROM. Tiap halaman tanggal mengembalikan link
    artikel hari itu; URL-nya memuat tanggal -> bisa menjangkau 5 tahun. Link yang
    judul/URL-nya cocok keyword di-tag kategori; sisanya '__PENDING__' (dicek via isi
    artikel saat parse) sampai batas anggaran -- sama seperti index mode biasa."""
    collected = {}
    per_cat_count = {}
    pending_budget = overall_cap * 3
    day = DATE_TO
    empty_streak = 0
    days_done = 0
    while day >= DATE_FROM and days_done < DATE_INDEX_MAX_DAYS:
        days_done += 1
        url_idx = DATE_INDEX_URL_FMT.format(yyyy=day.strftime('%Y'), mm=day.strftime('%m'), dd=day.strftime('%d'))
        resp = fetch(url_idx)
        polite_sleep()
        day = day - timedelta(days=1)
        if resp is None:
            empty_streak += 1
            if empty_streak >= 20:  # 20 hari beruntun gagal -> situs mungkin berubah; hentikan
                logger.warning('20 tanggal beruntun gagal diambil -- hentikan date-index.')
                break
            continue
        found = extract_links_with_titles(resp.text)
        if not found:
            empty_streak += 1
            continue
        empty_streak = 0
        added = 0
        for url, anchor in found.items():
            if url in collected:
                continue
            hits = match_keywords(f'{anchor} {url}')
            if hits:
                term, category = hits[0]
                if per_cat_count.get(category, 0) >= per_keyword_cap:
                    continue
                collected[url] = {'keywords': set(t for t, _ in hits), 'category': category}
                per_cat_count[category] = per_cat_count.get(category, 0) + 1
                added += 1
            elif pending_budget > 0:
                collected[url] = {'keywords': set(), 'category': '__PENDING__'}
                pending_budget -= 1
                added += 1
        logger.info('[date %s] +%d (total %d, pending_budget %d)', url_idx.rsplit('/indeks/', 1)[-1], added, len(collected), pending_budget)
        if len(collected) >= overall_cap * 4:
            return collected
    return collected

def collect_article_links(keywords, max_pages, per_keyword_cap, overall_cap):
    """Dispatch: DATE-INDEX (arsip per tanggal), INDEX MODE (robots.txt melarang /search), atau SEARCH MODE."""
    global CURRENT_KEYWORDS, CURRENT_IS_NEUTRAL
    CURRENT_KEYWORDS = keywords
    CURRENT_IS_NEUTRAL = (keywords is KEYWORDS_NEUTRAL) or (set(keywords) == set(KEYWORDS_NEUTRAL))
    if USE_DATE_INDEX:
        return collect_via_date_index(keywords, max_pages, per_keyword_cap, overall_cap)
    if USE_INDEX:
        return collect_via_index(keywords, max_pages, per_keyword_cap, overall_cap)
    return collect_via_search(keywords, max_pages, per_keyword_cap, overall_cap)

CURRENT_KEYWORDS = ACTIVE_KEYWORDS  # dipakai match_keywords() di index mode
CURRENT_IS_NEUTRAL = False
candidate_map = collect_article_links(ACTIVE_KEYWORDS, ACTIVE_MAX_PAGES, ACTIVE_PER_KEYWORD, ACTIVE_TARGET)
logger.info("Total kandidat URL unik: %d", len(candidate_map))
list(candidate_map.items())[:5]

2026-10-20 23:54:06,107 [INFO] [sampah menumpuk] page 1: +5 (total unik 5)
2026-10-20 23:54:08,101 [INFO] [tumpukan sampah] page 1: +5 (total unik 5)
2026-10-20 23:54:11,735 [INFO] [sampah berserakan] page 1: +2 (total unik 7)
2026-10-20 23:54:22,083 [INFO] [buang sampah sembarangan] page 1: +5 (total unik 12)
2026-10-20 23:54:27,649 [INFO] [jalan rusak] page 1: +5 (total unik 17)
2026-10-20 23:54:31,004 [INFO] [jalan berlubang] page 1: +5 (total unik 17)
2026-10-20 23:54:34,050 [INFO] [jalan amblas] page 1: +5 (total unik 17)
2026-10-20 23:54:44,200 [INFO] [jembatan rusak] page 1: +5 (total unik 17)
2026-10-20 23:55:10,740 [INFO] [fasilitas umum rusak] page 1: +5 (total unik 22)
2026-10-20 23:55:31,515 [INFO] [pkl liar] page 1: +5 (total unik 22)
2026-10-20 23:55:33,934 [INFO] [parkir liar] page 1: +5 (total unik 27)
2026-10-20 23:55:39,501 [INFO] [bangunan liar] page 1: +5 (total unik 27)
2026-10-20 23:55:42,305 [INFO] [juru parkir liar] page 1: +5 (total unik 27)
2026-10-20 23:55:45

[('https://news.okezone.com/read/2026/10/20/338/3248951/ada-26-titik-demo-hari-ini-polda-metro-hanya-dua-elemen-layangkan-surat-pemberitahuan-nbsp',
  {'keywords': {'bangunan liar',
    'banjir',
    'banjir kiriman',
    'banjir rob',
    'buang sampah sembarangan',
    'genangan air',
    'jalan amblas',
    'jalan berlubang',
    'juru parkir liar',
    'keluhan warga',
    'macet',
    'parkir liar',
    'pkl liar',
    'pungli',
    'sampah menumpuk',
    'tumpukan sampah',
    'warga keluhkan',
    'warga resah'},
   'category': 'Sampah'}),
 ('https://news.okezone.com/read/2026/10/20/338/3248937/demo-20-oktober-2-jenderal-bintang-3-terjun-langsung-siagakan-3-000-personel-gabungan-di-monas',
  {'keywords': {'bangunan liar',
    'banjir',
    'banjir kiriman',
    'banjir rob',
    'buang sampah sembarangan',
    'genangan air',
    'jalan amblas',
    'jalan berlubang',
    'juru parkir liar',
    'keluhan warga',
    'macet',
    'parkir liar',
    'pkl liar',
    'pungli',
    '

## 6. Parse Isi Artikel

`parse_article()` mengutamakan **JSON-LD** dan **meta tag** (`og:title`,
`article:published_time`, dsb.) karena lebih stabil daripada class CSS; selector CSS
dari `SELECTORS` dipakai sebagai fallback. Elemen non-artikel dibuang.

In [6]:
MONTHS_ID = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "mei": 5, "jun": 6,
    "jul": 7, "agu": 8, "ags": 8, "sep": 9, "okt": 10, "nov": 11, "des": 12,
}

def parse_date_any(text):
    """Parse ISO 8601 atau tanggal Indonesia -> datetime tz-aware (UTC bila tz tak ada)."""
    if not text:
        return None
    text = text.strip()
    # 1) Coba ISO 8601 (dari meta article:published_time / JSON-LD).
    try:
        dt = datetime.fromisoformat(text.replace("Z", "+00:00"))
        return dt if dt.tzinfo else dt.replace(tzinfo=timezone.utc)
    except ValueError:
        pass
    # 2) Tanggal Indonesia, mis. 'Senin, 12 Mei 2025 14:30 WIB'.
    m = re.search(r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})(?:[^\d]+(\d{1,2})[:.](\d{2}))?", text)
    if not m:
        return None
    day, mon, year = int(m.group(1)), m.group(2)[:3].lower(), int(m.group(3))
    month = MONTHS_ID.get(mon)
    if not month:
        return None
    hour = int(m.group(4)) if m.group(4) else 0
    minute = int(m.group(5)) if m.group(5) else 0
    try:
        return datetime(year, month, day, hour, minute, tzinfo=timezone.utc)
    except ValueError:
        return None

def parse_jsonld(soup):
    """Ambil blok JSON-LD tipe Article/NewsArticle bila ada."""
    for tag in soup.find_all('script', type='application/ld+json'):
        try:
            data = json.loads(tag.string or '{}')
        except (json.JSONDecodeError, TypeError):
            continue
        candidates = data if isinstance(data, list) else [data]
        if isinstance(data, dict) and '@graph' in data:
            candidates = data['@graph']
        for c in candidates:
            if isinstance(c, dict) and str(c.get('@type', '')).lower().endswith('article'):
                return c
    return {}

def clean_content(body_el):
    """Buang elemen non-artikel lalu gabungkan paragraf jadi teks bersih."""
    if body_el is None:
        return ""
    for sel in SELECTORS.get('junk', []):
        for junk in body_el.select(sel):
            junk.decompose()
    paragraphs = [p.get_text(' ', strip=True) for p in body_el.find_all('p')]
    text = ' '.join(p for p in paragraphs if p) or body_el.get_text(' ', strip=True)
    text = re.sub(r'\s+', ' ', text).strip()
    # Buang sisa 'Baca juga' / 'Simak juga'.
    text = re.sub(r'(Baca juga|Simak juga|ADVERTISEMENT|SCROLL TO CONTINUE)\s*:?', ' ', text, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).strip()

def build_article_url(url):
    """Tambahkan parameter buka-semua-halaman untuk artikel berhalaman (bila ada)."""
    if MULTIPAGE_ARTICLE and PAGE_ALL_PARAM:
        return url + PAGE_ALL_PARAM
    return url

def parse_article(url, keyword, keyword_category):
    """Ambil satu artikel -> dict sesuai OUTPUT_COLUMNS (+ _published_dt internal), atau None."""
    resp = fetch(build_article_url(url))
    polite_sleep()
    if resp is None:
        return None
    soup = BeautifulSoup(resp.text, 'lxml')
    ld = parse_jsonld(soup)

    # --- Judul: og:title / JSON-LD dulu, lalu selector CSS. ---
    title = (soup.select_one('meta[property="og:title"]') or {})
    title = title.get('content', '') if hasattr(title, 'get') else ''
    if not title:
        title = ld.get('headline', '') or select_first(soup, SELECTORS['title'])

    # --- Tanggal: meta article:published_time / JSON-LD datePublished, lalu CSS. ---
    date_meta = soup.select_one('meta[property="article:published_time"]')
    raw_date = date_meta.get('content', '') if date_meta else ''
    if not raw_date:
        raw_date = ld.get('datePublished', '')
    if not raw_date:
        raw_date = select_first(soup, SELECTORS['date'], attr='datetime') or select_first(soup, SELECTORS['date'])
    published = parse_date_any(raw_date)

    # --- Author: JSON-LD / meta, lalu CSS. ---
    author = ''
    if isinstance(ld.get('author'), dict):
        author = ld['author'].get('name', '')
    elif isinstance(ld.get('author'), list) and ld['author']:
        author = ld['author'][0].get('name', '') if isinstance(ld['author'][0], dict) else str(ld['author'][0])
    if not author:
        author = select_first(soup, ['meta[name="author"]'], attr='content') or select_first(soup, SELECTORS['author'])

    # --- Section / kanal. ---
    category_section = select_first(soup, SELECTORS['section'], attr='content')
    if not category_section and isinstance(ld.get('articleSection'), str):
        category_section = ld['articleSection']

    # --- Konten. ---
    body_el = None
    for sel in SELECTORS['body']:  # [CEK SAAT RUNNING]
        body_el = soup.select_one(sel)
        if body_el is not None:
            break
    content = clean_content(body_el)
    if not content and ld.get('articleBody'):
        content = re.sub(r'\s+', ' ', ld['articleBody']).strip()

    if not title or len(content) < MIN_CONTENT_CHARS:
        return None  # judul kosong atau konten terlalu pendek

    # --- Relevansi tahap-KONTEN (INDEX MODE): link yang dikumpulkan tanpa match judul
    #     di-tag '__PENDING__'. Di sini kita cek isi artikel penuh untuk menentukan kategori.
    if keyword_category == '__PENDING__':
        global CURRENT_IS_NEUTRAL
        prev_neutral = CURRENT_IS_NEUTRAL
        CURRENT_IS_NEUTRAL = False
        content_hits = match_keywords(f'{title} {content}')  # cek SELURUH isi artikel
        CURRENT_IS_NEUTRAL = prev_neutral
        if not content_hits:
            return {'_skip': 'not_relevant'}  # artikel terbaca, tapi bukan berita keluhan
        keyword, keyword_category = content_hits[0][0], content_hits[0][1]

    return {
        'source': SOURCE_NAME,
        'url': url,
        'title': title,
        'published_date': published.isoformat() if published else '',
        'category_section': category_section,
        'author': author,
        'content': content,
        'scraped_at': datetime.now(timezone.utc).isoformat(),
        'keyword_used': keyword,
        'keyword_category': keyword_category,
        '_published_dt': published,
    }

## 7. Pipeline Utama (dedup, filter, checkpoint, log error)

Loop kandidat URL: parse → filter tanggal (`DATE_FROM`..`DATE_TO`) & panjang min →
simpan checkpoint tiap `CHECKPOINT_EVERY`. Bisa **resume** (lewati URL yang sudah
ada di checkpoint). Error per artikel ditangkap & dicatat ke
`../data/raw/logs/<sumber>_errors.csv`.

In [7]:
def log_error(url, reason):
    """Catat URL gagal + alasan ke error-log CSV (append)."""
    new_file = not ERROR_LOG_PATH.exists()
    with open(ERROR_LOG_PATH, 'a', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(['url', 'reason', 'ts'])
        w.writerow([url, reason, datetime.now(timezone.utc).isoformat()])

def resume_from_checkpoint():
    """Muat checkpoint bila ada agar scraping bisa dilanjutkan."""
    if CHECKPOINT_PATH.exists():
        df_ckpt = pd.read_csv(CHECKPOINT_PATH)
        logger.info("Resume dari checkpoint: %d artikel.", len(df_ckpt))
        return df_ckpt.to_dict('records'), set(df_ckpt['url'].tolist())
    return [], set()

def scrape(candidate_items, target, label='complaint'):
    """Scrape daftar (url, keyword, category) sampai target artikel tercapai."""
    records, done_urls = resume_from_checkpoint()
    done_urls = set(done_urls)
    pbar = tqdm(candidate_items, desc=f'Scraping {SOURCE_NAME} [{label}]')
    for url, keyword, category in pbar:
        if len([r for r in records]) >= target:
            break
        if url in done_urls:
            continue
        try:
            article = parse_article(url, keyword, category)
        except Exception as exc:
            log_error(url, f'parse_exception: {exc}')
            continue
        if article is None:
            log_error(url, 'empty_or_short_or_no_title')
            continue
        if '_skip' in article:
            log_error(url, article['_skip'])  # mis. 'not_relevant' -- bukan error parsing
            done_urls.add(url)
            continue
        pub = article.pop('_published_dt', None)
        if pub is not None and not (DATE_FROM <= pub <= DATE_TO):
            log_error(url, f'out_of_date_range: {pub.date() if pub else None}')
            continue
        records.append(article)
        done_urls.add(url)
        pbar.set_postfix(ok=len(records))
        if len(records) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records).to_csv(CHECKPOINT_PATH, index=False, encoding='utf-8-sig')
    return records

# Bangun daftar kandidat (url, keyword_terpilih, category) dari candidate_map.
# keyword_used menggabungkan semua keyword asal dengan ' | '.
complaint_items = []
for url, meta in candidate_map.items():
    kws = sorted(meta['keywords'])
    complaint_items.append((url, ' | '.join(kws), meta['category']))

records = scrape(complaint_items, ACTIVE_TARGET, label='complaint')
logger.info("Artikel keluhan terkumpul: %d", len(records))

Scraping okezone [complaint]: 100%|██████████| 90/90 [03:55<00:00,  2.62s/it, ok=58]
2026-10-21 00:09:01,039 [INFO] Artikel keluhan terkumpul: 58


## 8. Kuota Artikel Netral (kelas 'Bukan Pengaduan')

Jalankan alur yang sama untuk `KEYWORDS_NEUTRAL` sampai kuota `NEUTRAL_RATIO`
terpenuhi, dengan `keyword_category = 'Neutral'`. Pada `TEST_MODE`, kuota = 0 (dilewati).

In [8]:
if ACTIVE_NEUTRAL_QUOTA > 0:
    neutral_map = collect_article_links(
        KEYWORDS_NEUTRAL,
        ACTIVE_MAX_PAGES,
        max(5, ACTIVE_NEUTRAL_QUOTA // max(1, len(KEYWORDS_NEUTRAL)) + 2),
        ACTIVE_NEUTRAL_QUOTA,
    )
    neutral_items = [
        (url, ' | '.join(sorted(meta['keywords'])), 'Neutral')
        for url, meta in neutral_map.items()
        if url not in {r['url'] for r in records}
    ]
    neutral_records = scrape(neutral_items, ACTIVE_NEUTRAL_QUOTA, label='neutral')
    # scrape() me-resume checkpoint lama, jadi ambil hanya yang kategori Neutral & belum ada.
    existing_urls = {r['url'] for r in records}
    for r in neutral_records:
        if r['url'] not in existing_urls and r['keyword_category'] == 'Neutral':
            records.append(r)
            existing_urls.add(r['url'])
    logger.info("Artikel netral ditambahkan. Total sekarang: %d", len(records))
else:
    logger.info("TEST_MODE / kuota netral 0 -> lewati scraping netral.")

2026-10-21 00:09:03,126 [INFO] [politik] page 1: +5 (total unik 5)
2026-10-21 00:09:05,183 [INFO] [olahraga] page 1: +5 (total unik 10)
2026-10-21 00:09:13,698 [INFO] [teknologi] page 1: +5 (total unik 10)
2026-10-21 00:09:16,561 [INFO] [kesehatan gaya hidup] page 1: +5 (total unik 10)
2026-10-21 00:09:25,255 [INFO] [otomotif] page 1: +1 (total unik 11)
2026-10-21 00:09:27,079 [INFO] [internasional] page 1: +5 (total unik 16)
2026-10-21 00:09:29,338 [INFO] [politik] page 2: +0 (total unik 16)
2026-10-21 00:09:32,042 [INFO] [olahraga] page 2: +5 (total unik 21)
2026-10-21 00:09:34,843 [INFO] [teknologi] page 2: +0 (total unik 21)
2026-10-21 00:09:41,163 [INFO] [otomotif] page 2: +5 (total unik 26)
2026-10-21 00:09:42,926 [INFO] [internasional] page 2: +5 (total unik 31)
2026-10-21 00:09:45,963 [INFO] [politik] page 3: +0 (total unik 31)
2026-10-21 00:09:47,742 [INFO] [olahraga] page 3: +1 (total unik 32)
2026-10-21 00:09:50,106 [INFO] [teknologi] page 3: +0 (total unik 32)
2026-10-21 00

## 9. Bersihkan, Dedup & Simpan CSV (utf-8-sig)

In [9]:
df = pd.DataFrame(records)

if df.empty:
    raise RuntimeError(
        'Tidak ada artikel terkumpul. Cek: (1) SELECTORS (sel Konfigurasi, tanda PERLU DIVERIFIKASI), '
        '(2) SEARCH_URL/SEARCH_PARAMS, (3) robots.txt/koneksi. '
        'Buka 1 URL pencarian & 1 URL artikel di browser untuk inspeksi selector.'
    )

# Pastikan semua kolom ada & urut.
for col in OUTPUT_COLUMNS:
    if col not in df.columns:
        df[col] = ''
df = df[OUTPUT_COLUMNS]

# Dedup berdasarkan URL lalu judul identik.
before = len(df)
df = df.drop_duplicates(subset=['url'])
df = df[df['title'].str.strip() != '']
df = df.drop_duplicates(subset=['title']).reset_index(drop=True)
# Buang konten kosong / terlalu pendek.
df = df[df['content'].fillna('').str.len() >= MIN_CONTENT_CHARS].reset_index(drop=True)
logger.info("Dedup & filter: %d -> %d artikel", before, len(df))

df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
if CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()  # hapus checkpoint setelah simpan final
print(f"Tersimpan {len(df)} artikel -> {OUTPUT_PATH}")

2026-10-21 00:11:04,493 [INFO] Dedup & filter: 70 -> 70 artikel


Tersimpan 70 artikel -> ..\data\raw\okezone_raw.csv


## 10. Ringkasan Hasil & Cek Kualitas

In [10]:
print(f"Sumber            : {SOURCE_NAME}")
print(f"Jumlah artikel    : {len(df)}")

dates = pd.to_datetime(df['published_date'], errors='coerce', utc=True).dropna()
if not dates.empty:
    print(f"Rentang tanggal   : {dates.min().date()}  ..  {dates.max().date()}")
else:
    print("Rentang tanggal   : (tanggal tidak terdeteksi untuk mayoritas artikel)")

df['content_len'] = df['content'].fillna('').str.len()
print(f"Rata-rata panjang : {df['content_len'].mean():.0f} karakter")

print("\nDistribusi keyword_category:")
print(df['keyword_category'].value_counts())

Sumber            : okezone
Jumlah artikel    : 70
Rentang tanggal   : 2024-11-12  ..  2026-10-20
Rata-rata panjang : 1479 karakter

Distribusi keyword_category:
keyword_category
Banjir                      15
Neutral                     12
Ketertiban Umum             11
General                     10
Transportasi/Lalu Lintas    10
Jalan                        7
Sampah                       5
Name: count, dtype: int64


In [11]:
# --- Cek kualitas sederhana ---
n = len(df)
empty_content = (df['content'].fillna('').str.len() < MIN_CONTENT_CHARS).sum()
empty_date = (df['published_date'].fillna('') == '').sum()
empty_title = (df['title'].fillna('').str.strip() == '').sum()
print("Cek kualitas:")
print(f"  Konten kosong/pendek : {empty_content} ({empty_content/n*100:.1f}%)")
print(f"  Tanggal kosong       : {empty_date} ({empty_date/n*100:.1f}%)")
print(f"  Judul kosong         : {empty_title} ({empty_title/n*100:.1f}%)")

# Error log (kalau ada).
if ERROR_LOG_PATH.exists():
    err = pd.read_csv(ERROR_LOG_PATH)
    print(f"\nURL gagal tercatat   : {len(err)} (lihat {ERROR_LOG_PATH})")

Cek kualitas:
  Konten kosong/pendek : 0 (0.0%)
  Tanggal kosong       : 0 (0.0%)
  Judul kosong         : 0 (0.0%)

URL gagal tercatat   : 166 (lihat ..\data\raw\logs\okezone_errors.csv)


In [12]:
# Contoh 5 baris.
df[['title', 'published_date', 'author', 'keyword_used', 'keyword_category']].head()

,title,published_date,author,keyword_used,keyword_category
0,"Ada 26 Titik Demo Hari Ini, Polda Metro: Hanya...",2026-10-20T10:45:00+07:00,Zakky Mubarok Santosa,bangunan liar | banjir | banjir kiriman | banj...,Sampah
1,"Demo 20 Oktober, 2 Jenderal Bintang 3 Terjun L...",2026-10-20T09:20:22+07:00,Zakky Mubarok Santosa,bangunan liar | banjir | banjir kiriman | banj...,Sampah
2,"Demo 20 Oktober, Polda Metro Akan Terapkan Rek...",2026-10-20T07:24:46+07:00,Ari Sandita Murti,aduan warga | bangunan liar | banjir | banjir ...,Sampah
3,Breaking News! Bharada AR Ditetapkan Tersangka...,2026-10-20T11:54:26+07:00,Zakky Mubarok Santosa,bangunan liar | banjir | banjir kiriman | banj...,Sampah
4,MNC Peduli Ajak Anak-Anak Gemar Membaca Lewat ...,2026-10-20T02:00:11+07:00,Nur Khabibi,bangunan liar | banjir | banjir kiriman | banj...,Sampah


In [13]:
# Cek isi konten 3 artikel pertama tidak kosong.
for i, row in df.head(3).iterrows():
    print(f"--- [{i}] {str(row['title'])[:80]} ---")
    print(str(row["content"])[:300], "...\n")

--- [0] Ada 26 Titik Demo Hari Ini, Polda Metro: Hanya Dua Elemen Layangkan Surat Pember ---
JAKARTA - Kabid Humas Polda Metro Jaya Kombes Pol Budi Hermanto mengungkapkan, dari 26 penyampaian aspirasi di muka umum hari ini, hanya ada dua elemen, yakni BEM UI dan Universitas Jayabaya, yang menyampaikan surat pemberitahuan pelaksanaan aksi kepada pihak kepolisian. "Ya, kami jawab. Dari 47 ele ...

--- [1] Demo 20 Oktober, 2 Jenderal Bintang 3 Terjun Langsung Siagakan 3.000 Personel Ga ---
JAKARTA - Sebanyak 3.000 ribu lebih personel gabungan dari unsur Polda Metro Jaya dan Kodam Jaya menggelar apel gabungan di Monumen Nasional (Monas), untuk persiapan pengamanan aksi demonstrasi di sejumlah titik Jakarta, pada Selasa, (20/10/2026). Pantauan Okezone di lokasi, mobil-mobil bus dari kep ...

--- [2] Demo 20 Oktober, Polda Metro Akan Terapkan Rekayasa Lalin Antisipasi Kemacetan : ---
JAKARTA - Polda Metro Jaya mengimbau masyarakat pengguna jalan untuk mengantisipasi potensi kepadatan lalu li

## 11. Hal yang Perlu Dicek Saat Running — Okezone

Selector di bawah **belum bisa diverifikasi tanpa menjalankan** (ditandai
`PERLU DIVERIFIKASI` / `# [CEK SAAT RUNNING]`). Kalau hasil kosong atau aneh,
periksa berurutan:

1. **`SEARCH_URL` & `SEARCH_PARAMS`** (sel Konfigurasi): buka URL pencarian di
   browser, lihat parameter asli di address bar, sesuaikan nama param/pagination.
2. **`SELECTORS['search_link']`**: kalau `candidate_map` kosong, Inspect Element
   pada daftar hasil pencarian untuk menemukan selector link artikel yang benar.
3. **`ARTICLE_URL_RE`**: pastikan pola URL artikel situs cocok (lihat 1 URL artikel asli).
4. **`SELECTORS['body']` / `title` / `date` / `author`**: kalau konten/metadata kosong,
   Inspect Element pada halaman artikel. Parser sudah mengutamakan JSON-LD & meta tag,
   jadi kemungkinan besar jalan walau class CSS berubah — tapi tetap konfirmasi.
5. **`robots.txt`**: kalau ada path DISALLOW, jangan paksakan; kurangi cakupan.